# Evaluation Notebook

This file performs evaluation of all algorithms/variants, stores results in a Zarr file, and generates plots.

## Configuration

In [1]:
# Set the model directory.
model_dir = "/data/group/mas/qmas/results/patrolling_zoo_v0/mappo/diffuser-patrolling/wandb/run-20250512_171642-l6ey1a3x/files"

# Set the output file.
# output_file = "output.zarr"

# Set the output file automatically.
output_file = "./" + model_dir.split("/")[-2] + ".zarr"

## Setup

In [2]:
%reload_ext autoreload
%autoreload 2
from onpolicy.scripts.eval.eval_pettingzoo import main

import os
import yaml
from copy import deepcopy
from multiprocessing import Process
os.environ["WANDB__SERVICE_WAIT"] = "300"

In [3]:
# Load arguments from the config file.
config_file = os.path.join(model_dir, "config.yaml")
args = yaml.load(open(config_file), Loader=yaml.FullLoader)

def convert_args(argsdict):
    # Convert arguments to the appropriate format.
    arglist = []
    for key, value in argsdict.items():
        if type(value) == dict and "value" in value and not key.startswith("_"):
            v = value["value"]
            if v == None:
                continue
            if type(v) == bool:
                if v:
                    arglist.append(f"--{key}")
                else:
                    arglist.append(f"--no-{key}")
            else:
                arglist.append(f"--{key}")
                arglist.append(str(v))
    return arglist

def wrap_arg_val(val):
    return {"value": val}

# Set required eval-specific arguments. Do not change these!
args["use_wandb"] = wrap_arg_val(False)
args["use_render"] = wrap_arg_val(False)
args["use_eval"] = wrap_arg_val(True)
args["model_dir"] = wrap_arg_val(model_dir)
args["cuda"] = wrap_arg_val(False)
args["eval_output_file"] = wrap_arg_val(output_file)

In [4]:
args["eval_episodes"] = wrap_arg_val(1)
args["episode_length"] = wrap_arg_val(10)
args["seed"] = wrap_arg_val(1)

# Evaluation

In [5]:
# Check if the output file already exists. If so, skip this cell.
if False and os.path.exists(args["eval_output_file"]["value"]):
    print("Output file already exists. Skipping evaluation.")

else:
    processes = []

    def start_process(args):
        # p = Process(target=main, args=[convert_args(args)])
        # processes.append(p)
        # p.start()
        main(convert_args(args))


    # VARIANT 1: Safety filtering with adaptive communication.
    argscopy = deepcopy(args)
    argscopy["experiment_name"] = wrap_arg_val("test1")
    start_process(argscopy)


    # VARIANT 2: ...
    #...


    # JOIN ALL PROCESSES
    for p in processes:
        p.join()
    print("All processes finished!")

Pettingzoo environment patrolling_zoo.patrolling_zoo_v0.parallel_env has additional arguments.
u are choosing to use mappo, we set use_recurrent_policy & use_naive_recurrent_policy to be False
Pettingzoo arguments validated: base
Pettingzoo environment patrolling_zoo.patrolling_zoo_v0.parallel_env has additional arguments validation function.
Pettingzoo arguments validated: patrolling_zoo.patrolling_zoo_v0.parallel_env
choose to use cpu...
PettingzooEnv: Attempting to pass unpacked argparse namespace to environment.
PettingzooEnv: Passing the following arguments to the environment: {'max_cycles': -1, 'num_agents': 4, 'alpha': 1.0, 'beta': 0.5, 'graph_file': '../patrolling_zoo/graphs/milwaukee.graph', 'graph_random': False, 'graph_random_nodes': 40, 'reward_method_terminal': 'average', 'reward_interval': -1, 'agent_speed': 40.0, 'action_method': 'neighbors', 'observe_method': 'adjacency', 'observe_method_global': 'adjacency', 'observe_bitmap_size': 50, 'observation_radius': 999999.0, 'a

/home/anthony/miniconda3/envs/qmas2/lib/python3.11/site-packages/torchrl/__init__.py:36: UserWarning: failed to set start method to spawn, and current start method for mp is fork.
  warn(


[ models/temporal ] Channel dimensions: [(1657, 256), (256, 128), (128, 64), (64, 32)]
[(1657, 256), (256, 128), (128, 64), (64, 32)]
[(1657, 32), (32, 64), (64, 128), (128, 256)]


/data/anthony/dev/qmas_task_allocation-1/onpolicy/models/diffusion/diffusion.py:86: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  betas * np.sqrt(alphas_cumprod_prev) / (1. - alphas_cumprod))


QmasAlgorithm: Null value for environment is -1.0.
Buffer initialized with episode length 10


ValueError: Invalid action 2 for agent agent_2. Node 2 has only 1 neighbors.

# Analysis

In [ ]:
import zarr
output = zarr.open(output_file, mode='r')
output.tree()

Tree(nodes=(Node(disabled=True, name='/', nodes=(Node(disabled=True, name='test1', opened=False),)),))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Calculate time to goal.
time_to_goal = np.zeros((len(output), all_args.eval_episodes))
for i, experiment in enumerate(output):
    sig = output[experiment]['sheep_in_goal']
    for j in range(sig.shape[0]):
        all_in_goal = np.argwhere(sig[j] >= all_args.num_sheep)
        if len(all_in_goal) > 0:
            all_in_goal = all_in_goal[0][0]
        else:
            all_in_goal = all_args.episode_length
        time_to_goal[i, j] = all_in_goal

# Data for plotting
categories = {
    'communicated_agents': {
        'title': 'Messages Sent',
        'ylabel': 'Avg. Messages Sent (All Agents)',
        'data': [np.mean(np.sum(output[experiment]['communicated_agents'], axis=1)) for experiment in output]
    },
    'sheep_in_danger': {
        'title': 'Time in Danger',
        'ylabel': 'Avg. Timesteps in Danger (Cumulative, All Sheep)',
        'data': [np.mean(np.sum(output[experiment]['sheep_in_danger'], axis=1)) for experiment in output]
    },
    'sheep_in_goal': {
        'title': 'Time in Goal',
        'ylabel': 'Avg. Timesteps in Goal (Cumulative, All Sheep)',
        'data': [np.mean(np.sum(output[experiment]['sheep_in_goal'], axis=1)) for experiment in output]
    },
    'time_to_goal': {
        'title': 'Time to Goal',
        'ylabel': 'Avg. Timesteps to All Sheep in Goal',
        'data': [np.mean(time_to_goal[i]) for i in range(len(output))]
    }
}
for category in categories:
    values = categories[category]['data']

    # Create bar chart
    plt.bar(list(output.keys()), values)
    
    title = categories[category]['title']
    plt.xticks(rotation=45, ha='right')
    plt.xlabel('Scenario' if "xlabel" not in categories[category] else categories[category]['xlabel'])
    plt.ylabel(f"Timesteps out of {all_args.episode_length}" if "ylabel" not in categories[category] else categories[category]['ylabel'])
    plt.title(f'{title}: avg. over {all_args.eval_episodes} episodes, seed {all_args.seed}')
    plt.show()

NameError: name 'all_args' is not defined